In [5]:
import pandas as pd
import sqlite3
import os

In [9]:
def csv_to_sqlite(csv_path, db_name='catalog.db', table_name='user_catalog', if_exists='replace'):
    """
    Read a CSV file and store its contents in a SQLite table.
    
    Parameters:
    csv_path (str): Path to the CSV file.
    db_name (str): Name of the SQLite database file (default: 'catalog.db').
    table_name (str): Name of the table to create or replace (default: 'user_catalog').
    if_exists (str): Behavior if the table exists. Options: 'fail', 'replace', 'append' (default: 'replace').
    
    Returns:
    None
    """
    try:
        # Read the CSV file
        df = pd.read_csv(csv_path)
        print(f"Successfully read CSV file: {csv_path}")
        print(f"DataFrame shape: {df.shape}")
        
        # Connect to SQLite database (creates it if it doesn't exist)
        conn = sqlite3.connect(db_name)
        print(f"Connected to database: {db_name}")
        
        # Write the DataFrame to SQLite table
        df.to_sql(table_name, conn, if_exists=if_exists, index=False)
        print(f"Data inserted into table '{table_name}' with if_exists='{if_exists}'.")
        
        # Verify by reading back a few rows
        cursor = conn.cursor()
        cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
        rows = cursor.fetchall()
        print(f"First 3 rows from {table_name}:\n{rows}")
        
        # Close the connection
        conn.close()
        print("Database connection closed.")
    except FileNotFoundError:
        print(f"Error: The file {csv_path} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")


In [10]:
import requests\n
from typing import Union\n
\n
\n
def fetch_best_book_match(title: str, author: str) -> Union[dict, None]:\n
# Check if the file exists (optional)
if os.path.exists(csv_file_path):
    csv_to_sqlite(csv_file_path)
else:
    print(f"Please update the csv_file_path variable to point to your CSV file.\nCurrent path '{csv_file_path}' does not exist.")


Successfully read CSV file: goodreads-export-sample.csv
DataFrame shape: (8, 24)
Connected to database: catalog.db
Data inserted into table 'user_catalog' with if_exists='replace'.
First 3 rows from user_catalog:
[(58213401, "Mona's Eyes", 'Thomas Schlesser', 'Schlesser, Thomas', None, '="1643756033"', '="9781643756031"', 5, 4.32, 'Europa Editions', 'Hardcover', 354, 2025, 2024, '2026/09/27', '2026/09/20', 'favorites', 'favorites (#1)', 'read', None, None, None, 1, 0), (29184756, 'The Nightingale', 'Kristin Hannah', 'Hannah, Kristin', None, '="0312577222"', '="9780312577223"', 5, 4.57, "St. Martin's Press", 'Hardcover', 440, 2015, 2015, '2026/03/21', '2026/03/02', None, None, 'read', None, None, None, 1, 1), (37482910, 'How to Hide an Empire: A History of the Greater United States', 'Daniel Immerwahr', 'Immerwahr, Daniel', None, '="0374172146"', '="9780374172145"', 5, 4.34, 'Farrar, Straus and Giroux', 'Hardcover', 528, 2019, 2019, '2026/06/28', '2026/06/10', None, None, 'read', None, 

In [11]:
# Inspect the SQLite table: column names and first few rows
import sqlite3
import pandas as pd

conn = sqlite3.connect('catalog.db')

# Show column names
cursor = conn.cursor()
cursor.execute("PRAGMA table_info(user_catalog)")
columns = cursor.fetchall()
print("Column names:")
for col in columns:
    print(f"  [{col[0]}] {col[1]} ({col[2]})")

print()

# Preview first 5 rows as a DataFrame
df_preview = pd.read_sql_query("SELECT * FROM user_catalog LIMIT 5", conn)
conn.close()

df_preview


Column names:
  [0] Book Id (INTEGER)
  [1] Title (TEXT)
  [2] Author (TEXT)
  [3] Author l-f (TEXT)
  [4] Additional Authors (TEXT)
  [5] ISBN (TEXT)
  [6] ISBN13 (TEXT)
  [7] My Rating (INTEGER)
  [8] Average Rating (REAL)
  [9] Publisher (TEXT)
  [10] Binding (TEXT)
  [11] Number of Pages (INTEGER)
  [12] Year Published (INTEGER)
  [13] Original Publication Year (INTEGER)
  [14] Date Read (TEXT)
  [15] Date Added (TEXT)
  [16] Bookshelves (TEXT)
  [17] Bookshelves with positions (TEXT)
  [18] Exclusive Shelf (TEXT)
  [19] My Review (REAL)
  [20] Spoiler (REAL)
  [21] Private Notes (REAL)
  [22] Read Count (INTEGER)
  [23] Owned Copies (INTEGER)



,Book Id,Title,Author,Author l-f,Additional Authors,ISBN,ISBN13,My Rating,Average Rating,Publisher,...,Date Read,Date Added,Bookshelves,Bookshelves with positions,Exclusive Shelf,My Review,Spoiler,Private Notes,Read Count,Owned Copies
0,58213401,Mona's Eyes,Thomas Schlesser,"Schlesser, Thomas",None,"=""1643756033""","=""9781643756031""",5,4.32,Europa Editions,...,2026/09/27,2026/09/20,favorites,favorites (#1),read,None,None,None,1,0
1,29184756,The Nightingale,Kristin Hannah,"Hannah, Kristin",None,"=""0312577222""","=""9780312577223""",5,4.57,St. Martin's Press,...,2026/03/21,2026/03/02,None,None,read,None,None,None,1,1
2,37482910,How to Hide an Empire: A History of the Greate...,Daniel Immerwahr,"Immerwahr, Daniel",None,"=""0374172146""","=""9780374172145""",5,4.34,"Farrar, Straus and Giroux",...,2026/06/28,2026/06/10,None,None,read,None,None,None,1,0
3,44829173,Lock Every Door,Riley Sager,"Sager, Riley",None,"=""1524745146""","=""9781524745141""",5,3.94,Dutton,...,2026/07/19,2026/07/05,None,None,read,None,None,None,1,0
4,51029384,Hooked,Asako Yuzuki,"Yuzuki, Asako",Polly Barton,"=""0063317008""","=""9780063317009""",4,3.88,HarperVia,...,2026/05/17,2026/05/01,None,None,read,None,None,None,1,0


In [21]:
import requests
import re
from typing import Union

# ── Priority Genres & Themes ─────────────────────────────────────────────────
PRIORITY_GENRES = [
    "historical fiction",
    "science fiction",
    "fiction",
    "nonfiction",
    "romance",
    "thriller",
    "mystery",
    "suspense",
    "war",
    "family",
    "art",
    "history",
    "biography",
    "autobiography",
    "memoir",
    "science",
    "fantasy",
    "crime",
    "horror",
    "adventure",
    "poetry",
    "drama",
    "philosophy",
    "psychology",
    "politics",
]


def clean_and_prioritize_subjects(
    raw_subjects: list,
    places: list = None,
    times: list = None,
    limit: int = 5,
) -> list:
    """
    Filter, clean, and prioritize book subjects:
      1. Convert to lowercase & deduplicate (preserving order).
      2. Drop any subject containing ":".
      3. Drop place and time tags.
      4. Prioritize genre and theme tags (even if beyond the first 5 in raw subjects).
      5. Drop marketing tags (bestseller, reviews) and redundant combinations.
      6. Return up to limit clean subjects.
    """
    if not raw_subjects:
        return []

    places_set = {str(p).lower().strip() for p in (places or []) if p}
    times_set = {str(t).lower().strip() for t in (times or []) if t}

    # Temporal patterns: years (e.g. 1939-1945, 1940s), centuries, or era markers
    time_regex = re.compile(
        r"((1\d{3}|20\d{2})s?|\d{1,2}(st|nd|rd|th)\s+century)",
        re.IGNORECASE,
    )

    seen = set()
    cleaned = []

    for item in raw_subjects:
        if not item or not isinstance(item, str):
            continue

        s = item.lower().strip()

        # Drop anything containing ":"
        if ":" in s:
            continue

        # Drop library catalog URIs or fast / OCLC identifiers
        if "http" in s or "fast (" in s or "(ocolc)" in s:
            continue

        # Drop review / bestseller / marketing tags
        if "new york times" in s or "bestseller" in s:
            continue

        # Deduplicate
        if s in seen:
            continue
        seen.add(s)

        # Drop place and time tags
        if s in places_set or s in times_set:
            continue
        if "(imaginary place)" in s or "(place)" in s:
            continue
        if any(p in s for p in places_set if len(p) > 2) and (
            ", fiction" in s or "description and travel" in s
        ):
            continue
        if time_regex.search(s) and not any(kw in s for kw in ["war", "history"]):
            continue

        cleaned.append(s)

    # Prioritize:
    # Tier 1: exact match to a priority genre/theme
    # Tier 2: contains a priority genre/theme as a whole word / phrase
    # Tier 3: any other valid subjects
    tier1 = []
    tier2 = []
    tier3 = []

    for s in cleaned:
        if s in PRIORITY_GENRES:
            tier1.append(s)
        elif any(re.search(r"" + re.escape(g) + r"", s) for g in PRIORITY_GENRES):
            tier2.append(s)
        else:
            tier3.append(s)

    # Assemble without redundant combinations of already selected words
    selected = []
    selected_words = set()

    for s in (tier1 + tier2 + tier3):
        words = set(re.findall(r"\w+", s))
        if len(words) > 1 and words.issubset(selected_words):
            continue
        selected.append(s)
        selected_words.update(words)
        if len(selected) >= limit:
            break

    return selected


def fetch_best_book_match(title: str, author: str) -> Union[dict, None]:
    """
    Query the Open Library Search API for a book by title and author.
    Returns the result with the most editions — i.e. the canonical work.
    Returns None if no results are found.
    """
    url = "https://openlibrary.org/search.json"
    params = {
        "title": title,
        "author": author,
        "limit": 3,
        "fields": ",".join([
            "key",
            "title",
            "author_name",
            "first_publish_year",
            "edition_count",
            "number_of_pages_median",
            "ratings_average",
            "ratings_count",
            "subject",
            "place",
            "time",
            "cover_i",
        ]),
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    docs = response.json().get("docs", [])

    if not docs:
        return None

    # Pick the result with the highest edition_count as the canonical match
    best = max(docs, key=lambda d: d.get("edition_count", 0))
    return best


def format_result(doc: dict, fallback_pages: int = None) -> dict:
    """
    Flatten an Open Library doc into a clean dict for display.
    
    Includes edge-case handling for page counts:
      - Uses Open Library's `number_of_pages_median`.
      - If the median is missing (None) or an extreme low outlier (< 25 pages,
        such as a teaser/pamphlet), falls back to `fallback_pages` (e.g. from Goodreads).
    """
    subjects = clean_and_prioritize_subjects(
        raw_subjects=doc.get("subject"),
        places=doc.get("place"),
        times=doc.get("time"),
        limit=5,
    )

    # Validate pages_median & protect against low outliers (e.g. 12-page samplers)
    raw_pages = doc.get("number_of_pages_median")
    pages_median = None
    if raw_pages is not None:
        try:
            val = int(raw_pages)
            # If val < 25 and we have a realistic fallback, use fallback
            if val < 25 and fallback_pages and fallback_pages >= 25:
                pages_median = int(fallback_pages)
            else:
                pages_median = val
        except (ValueError, TypeError):
            pages_median = int(fallback_pages) if fallback_pages else None
    elif fallback_pages:
        try:
            pages_median = int(fallback_pages)
        except (ValueError, TypeError):
            pages_median = None

    return {
        "ol_key":             doc.get("key"),
        "ol_title":           doc.get("title"),
        "ol_author":          ", ".join(doc.get("author_name") or []),
        "first_publish_year": doc.get("first_publish_year"),
        "edition_count":      doc.get("edition_count"),
        "pages_median":       pages_median,
        "ratings_average":    round(doc.get("ratings_average") or 0, 2),
        "ratings_count":      doc.get("ratings_count"),
        "cover_id":           doc.get("cover_i"),
        "cover_url":          f"https://covers.openlibrary.org/b/id/{doc['cover_i']}-M.jpg"
                              if doc.get("cover_i") else None,
        "subjects":           ", ".join(subjects),
    }


In [22]:
# ── Test ─────────────────────────────────────────────────────────────────────
test_title  = "happy place"
test_author = "emily henry"

doc = fetch_best_book_match(test_title, test_author)

if doc:
    result = format_result(doc)
    print(f"Best match for '{test_title}' by {test_author}:")
    for k, v in result.items():
        print(f"  {k:<22} {v}")
else:
    print("No match found.")

Best match for 'happy place' by emily henry:
  ol_key                 /works/OL28607665W
  ol_title               Happy Place
  ol_author              Emily Henry
  first_publish_year     2023
  edition_count          9
  pages_median           400
  ratings_average        3.73
  ratings_count          15
  cover_id               13233403
  cover_url              https://covers.openlibrary.org/b/id/13233403-M.jpg
  subjects               
